# Implementación de los tres procedimientos sobre el test congelado

**INF-8267-C1 · Selección y Validación de Modelos · Asignación Semana 2 – Caso 4: Constructora Horizonte**

**Integrante 3 – Félix Manuel Pérez Abreu**

Este notebook re-ejecuta los tres procedimientos que el protocolo congelado de la Semana 1 dejó definidos, y comprueba mediante asserts que los tres se evalúan sobre exactamente el mismo test, con las mismas filas, el mismo target y sin fuga de `engineer_estimate_days` hacia las features.

1. **Baseline operativo** – estimación del ingeniero (`engineer_estimate_days`)
2. **Modelo mínimo** – Regresión Lineal con preprocesador ajustado únicamente con train
3. **Modelo alternativo** – Random Forest con los hiperparámetros congelados (`n_estimators=300, max_depth=8, min_samples_leaf=10, random_state=2026`)

El objetivo NO es cambiar el protocolo ni volver a seleccionar modelos, sino auditar la reproducibilidad y verificar que la comparación es limpia (mismo test, mismas filas, sin fuga).

In [1]:
import hashlib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings("ignore")

SEED = 2026
np.random.seed(SEED)

## 1. Carga de las particiones congeladas

Se cargan los archivos `train.csv`, `validacion.csv` y `test.csv` tal como los compartió el grupo para Semana 2. No se regeneran ni se re-particionan.

In [2]:
# Subir estos archivos antes de correr esta celda:
# train.csv, validacion.csv, test.csv
RUTA_TRAIN = "train.csv"
RUTA_VAL = "validacion.csv"
RUTA_TEST = "test.csv"

TARGET = "days_remaining"
COL_FECHA = "week_date"
COL_ENTIDAD = "workfront_id"
COL_BASELINE_OPERATIVO = "engineer_estimate_days"

VARIABLES_FUGA = [
    "final_completion_date", "total_final_cost",
    "final_delay_reason", "final_progress_report",
]
IDENTIFICADORES = ["project_id", "workfront_id", "contractor_id"]

train = pd.read_csv(RUTA_TRAIN, parse_dates=[COL_FECHA])
val = pd.read_csv(RUTA_VAL, parse_dates=[COL_FECHA])
test = pd.read_csv(RUTA_TEST, parse_dates=[COL_FECHA])

for nombre, df in [("train", train), ("validacion", val), ("test", test)]:
    df.drop(columns=[c for c in VARIABLES_FUGA if c in df.columns], inplace=True)
    print(f"{nombre}: {len(df):,} filas, {df[COL_ENTIDAD].nunique()} frentes, "
          f"{df[COL_FECHA].min().date()} a {df[COL_FECHA].max().date()}")

train: 2,805 filas, 150 frentes, 2025-01-19 a 2025-06-30
validacion: 1,092 filas, 128 frentes, 2025-07-01 a 2025-08-31
test: 1,273 filas, 119 frentes, 2025-09-01 a 2025-12-11


## 2. Corrección determinista del target

El protocolo congelado establece `days_remaining ≤ 2 × days_elapsed` (por frente, mínimo 14) como mitigación del defecto conocido del generador. Se aplica exactamente igual a train, validación y test.

In [3]:
def aplicar_tope_target(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    df = df.copy()
    tope = df.groupby(COL_ENTIDAD)["days_elapsed"].transform("max") * 2
    tope = tope.clip(lower=14)
    n_afectados = int((df[TARGET] > tope).sum())
    if n_afectados > 0:
        pct = n_afectados / len(df) * 100
        print(f"[{nombre}] Corregidas {n_afectados} filas ({pct:.1f}%) "
              f"por el defecto conocido del generador.")
    df[TARGET] = np.minimum(df[TARGET], tope)
    return df

train = aplicar_tope_target(train, "train")
val = aplicar_tope_target(val, "validacion")
test = aplicar_tope_target(test, "test")

[train] Corregidas 1628 filas (58.0%) por el defecto conocido del generador.
[validacion] Corregidas 602 filas (55.1%) por el defecto conocido del generador.
[test] Corregidas 432 filas (33.9%) por el defecto conocido del generador.


## 3. Huella del test antes de tocarlo

Se calcula un hash SHA-256 sobre las columnas de test que participan en la evaluación. Esa huella se re-verifica al final del notebook: si algo del test cambió entre procedimientos, la huella cambiaría.

In [4]:
FEATURES_NUMERICAS = [
    "planned_progress_pct", "actual_progress_pct", "days_elapsed",
    "workers_assigned", "equipment_hours", "material_availability_pct",
    "rain_days_last_week", "change_orders_to_date",
]
FEATURES_CATEGORICAS = ["work_type", "province"]
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS

def hash_dataframe(df: pd.DataFrame) -> str:
    b = pd.util.hash_pandas_object(df, index=True).values.tobytes()
    return hashlib.sha256(b).hexdigest()[:16]

cols_huella = [COL_FECHA, COL_ENTIDAD, TARGET, COL_BASELINE_OPERATIVO] + FEATURES
huella_test_inicial = hash_dataframe(test[cols_huella])
indice_test_inicial = test.index.copy()
target_test_inicial = test[TARGET].copy()

print(f"Huella SHA-256 del test tras aplicar el tope: {huella_test_inicial}")
print(f"Filas de test: {len(test):,}   |   frentes: {test[COL_ENTIDAD].nunique()}")

Huella SHA-256 del test tras aplicar el tope: 40e463d204777e03
Filas de test: 1,273   |   frentes: 119


## 4. Asserts de auditoría sobre las features

Antes de entrenar cualquier modelo se comprueba que las features definidas por el protocolo NO incluyen:
- `engineer_estimate_days` (baseline operativo, no debe entrar como feature)
- las 4 variables de fuga
- los identificadores

In [5]:
assert COL_BASELINE_OPERATIVO not in FEATURES, \
    "engineer_estimate_days NO puede estar en las features del modelo."
for col in VARIABLES_FUGA:
    assert col not in FEATURES, f"Variable de fuga presente en features: {col}"
for col in IDENTIFICADORES:
    assert col not in FEATURES, f"Identificador presente en features: {col}"

print("OK: engineer_estimate_days fuera de features.")
print("OK: 4 variables de fuga fuera de features.")
print("OK: 3 identificadores fuera de features.")
print(f"Features usadas ({len(FEATURES)}): {FEATURES}")

OK: engineer_estimate_days fuera de features.
OK: 4 variables de fuga fuera de features.
OK: 3 identificadores fuera de features.
Features usadas (10): ['planned_progress_pct', 'actual_progress_pct', 'days_elapsed', 'workers_assigned', 'equipment_hours', 'material_availability_pct', 'rain_days_last_week', 'change_orders_to_date', 'work_type', 'province']


## 5. Preprocesador ajustado únicamente con train

El mismo `preprocesador` (imputación de mediana para numéricas, moda + one-hot para categóricas) se usa dentro del pipeline de la Regresión Lineal y de Random Forest. Al estar dentro de un `Pipeline` de sklearn se garantiza que el `fit` ocurre solo sobre train y que el `transform` se aplica sin refit a validación y test.

In [6]:
preprocesador = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), FEATURES_NUMERICAS),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), FEATURES_CATEGORICAS),
])

## 6. Procedimiento A – Baseline operativo (ingeniero)

La predicción es directamente `engineer_estimate_days` tal como viene en test. No requiere entrenamiento.

In [7]:
pred_ingeniero = test[COL_BASELINE_OPERATIVO].values.copy()
print(f"Predicciones del ingeniero listas: {len(pred_ingeniero)} valores.")

Predicciones del ingeniero listas: 1273 valores.


## 7. Procedimiento B – Modelo mínimo (Regresión Lineal)

In [8]:
modelo_lineal = Pipeline([
    ("prep", preprocesador),
    ("reg", LinearRegression()),
])
modelo_lineal.fit(train[FEATURES], train[TARGET])
pred_lineal = np.clip(modelo_lineal.predict(test[FEATURES]), 0, None)
print(f"Modelo mínimo entrenado y evaluado sobre test: {len(pred_lineal)} predicciones.")

Modelo mínimo entrenado y evaluado sobre test: 1273 predicciones.


## 8. Procedimiento C – Modelo alternativo (Random Forest)

In [9]:
modelo_rf = Pipeline([
    ("prep", preprocesador),
    ("reg", RandomForestRegressor(
        n_estimators=300, max_depth=8, min_samples_leaf=10,
        random_state=SEED, n_jobs=-1,
    )),
])
modelo_rf.fit(train[FEATURES], train[TARGET])
pred_rf = np.clip(modelo_rf.predict(test[FEATURES]), 0, None)
print(f"Modelo alternativo entrenado y evaluado sobre test: {len(pred_rf)} predicciones.")

Modelo alternativo entrenado y evaluado sobre test: 1273 predicciones.


## 9. Asserts de auditoría post-modelado

Se verifica que:

- La huella SHA-256 del test **no cambió** entre el inicio del notebook y este punto (ningún procedimiento reprocesó el test).
- El índice y el vector de target del test son idénticos a los originales.
- Los tres vectores de predicción tienen exactamente la misma longitud que el test.
- El `SimpleImputer` numérico dentro del preprocesador se ajustó con la mediana de train, no de train+val+test.
- El `OneHotEncoder` conoce solo categorías presentes en train.

In [10]:
huella_test_final = hash_dataframe(test[cols_huella])
assert huella_test_final == huella_test_inicial, \
    "El test cambió entre procedimientos - se rompió la comparabilidad."
assert list(test.index) == list(indice_test_inicial), "El índice del test cambió."
assert (test[TARGET].values == target_test_inicial.values).all(), \
    "El target del test cambió."
assert len(pred_ingeniero) == len(pred_lineal) == len(pred_rf) == len(test), \
    "Los tres procedimientos deben producir len(pred) == len(test)."

imp_num = modelo_rf.named_steps["prep"].named_transformers_["num"]
medianas_train = train[FEATURES_NUMERICAS].median().values
np.testing.assert_allclose(
    imp_num.statistics_, medianas_train, rtol=1e-9,
    err_msg="El SimpleImputer se ajustó con datos distintos a train.",
)

imp_cat = modelo_rf.named_steps["prep"].named_transformers_["cat"]
ohe = imp_cat.named_steps["onehot"]
for i, c in enumerate(FEATURES_CATEGORICAS):
    cats_train = set(train[c].dropna().unique())
    cats_ohe = set(ohe.categories_[i])
    assert cats_ohe.issubset(cats_train | {train[c].mode().iloc[0]}), \
        f"El OHE conoce categorías no vistas en train para {c}: {cats_ohe - cats_train}"

print("OK: huella del test IDÉNTICA antes y después de correr los 3 procedimientos.")
print(f"    Huella final: {huella_test_final}")
print("OK: índice del test intacto.")
print("OK: target del test intacto.")
print("OK: SimpleImputer numérico ajustado ÚNICAMENTE con la mediana de train.")
print("OK: OneHotEncoder ajustado con categorías de train.")

OK: huella del test IDÉNTICA antes y después de correr los 3 procedimientos.
    Huella final: 40e463d204777e03
OK: índice del test intacto.
OK: target del test intacto.
OK: SimpleImputer numérico ajustado ÚNICAMENTE con la mediana de train.
OK: OneHotEncoder ajustado con categorías de train.


## 10. Métricas comparativas – mismo test para los tres procedimientos

MAE (métrica principal del protocolo), RMSE, porcentaje de subestimaciones, MAE de subestimaciones y MAE de sobrestimaciones. El desglose asimétrico es relevante porque en este problema subestimar es más caro.

In [11]:
def calcular_metricas(y_real, y_pred, nombre):
    error = pd.Series(y_pred).reset_index(drop=True) - pd.Series(y_real).reset_index(drop=True)
    mae = error.abs().mean()
    rmse = np.sqrt((error ** 2).mean())
    subestimaciones = (error < 0)
    return {
        "procedimiento": nombre,
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "pct_subestimaciones": round(subestimaciones.mean(), 3),
        "mae_subestimacion": round(error[subestimaciones].abs().mean(), 2)
            if subestimaciones.any() else np.nan,
        "mae_sobrestimacion": round(error[~subestimaciones].abs().mean(), 2)
            if (~subestimaciones).any() else np.nan,
    }

resultados = pd.DataFrame([
    calcular_metricas(test[TARGET], pred_ingeniero, "Baseline operativo (ingeniero)"),
    calcular_metricas(test[TARGET], pred_lineal, "Modelo mínimo (Regresión Lineal)"),
    calcular_metricas(test[TARGET], pred_rf, "Modelo alternativo (Random Forest)"),
])
print(resultados.to_string(index=False))

                     procedimiento    mae   rmse  pct_subestimaciones  mae_subestimacion  mae_sobrestimacion
    Baseline operativo (ingeniero) 121.76 238.54                0.436              27.21              195.89
  Modelo mínimo (Regresión Lineal) 143.43 159.24                0.610             150.93              131.71
Modelo alternativo (Random Forest) 131.23 169.25                0.764             162.35               30.75


## 11. Mejora porcentual sobre el ingeniero

El protocolo congelado establece un umbral de **15%** de mejora mínima en MAE frente al baseline operativo para recomendar reemplazarlo por el modelo.

In [12]:
mae_ing = resultados.loc[0, "mae"]
mae_lin = resultados.loc[1, "mae"]
mae_rf = resultados.loc[2, "mae"]
mejora_lin = (mae_ing - mae_lin) / mae_ing * 100
mejora_rf = (mae_ing - mae_rf) / mae_ing * 100
print(f"MAE ingeniero        : {mae_ing:.2f} días")
print(f"MAE Regresión Lineal : {mae_lin:.2f} días  (mejora {mejora_lin:+.1f}% sobre el ingeniero)")
print(f"MAE Random Forest    : {mae_rf:.2f} días  (mejora {mejora_rf:+.1f}% sobre el ingeniero)")
print("Umbral definido en el protocolo congelado para recomendar reemplazo: 15%.")

MAE ingeniero        : 121.76 días
MAE Regresión Lineal : 143.43 días  (mejora -17.8% sobre el ingeniero)
MAE Random Forest    : 131.23 días  (mejora -7.8% sobre el ingeniero)
Umbral definido en el protocolo congelado para recomendar reemplazo: 15%.


## 12. Cotejo con los resultados de Semana 1 y hallazgo de auditoría

El notebook `Modelado.ipynb` de la Semana 1 reportó, sobre el mismo protocolo, MAE 242.85 para el ingeniero, 176.60 para la Regresión Lineal y 159.11 para Random Forest.

La re-ejecución sobre las particiones congeladas compartidas para Semana 2 produce números notablemente distintos. Al investigar la causa se confirmó que:

- Los archivos `train.csv / validacion.csv / test.csv` compartidos en la carpeta "Archivos_ Dataset y Partición de Semana 1" fueron regenerados a partir del dataset **OLD** (`Etapa 2 - Generar datos/OLD/`), con 5,170 filas totales.
- Los números finales de Semana 1 provienen del dataset **08082026** (`constructora_horizonte_dias_restantes_08082026.csv`), con 5,728 filas totales.
- Re-particionando el 08082026 con la misma regla temporal se reproducen exactamente los MAE de Semana 1 (242.85, 176.60, 159.11).

En consecuencia, los tres procedimientos se implementan y comparan aquí sobre el test que hoy tiene el grupo como "congelado", pero **la comparación numérica no puede juntarse con los MAE reportados en Semana 1 sin advertir que provienen de un dataset distinto**.

In [13]:
cotejo = pd.DataFrame({
    "Procedimiento": ["Baseline operativo (ingeniero)",
                       "Modelo mínimo (Regresión Lineal)",
                       "Modelo alternativo (Random Forest)"],
    "MAE Semana 1 (dataset 08082026)": [242.85, 176.60, 159.11],
    "MAE re-ejecución (test congelado compartido)": [
        mae_ing, mae_lin, mae_rf,
    ],
})
print(cotejo.to_string(index=False))

                     Procedimiento  MAE Semana 1 (dataset 08082026)  MAE re-ejecución (test congelado compartido)
    Baseline operativo (ingeniero)                           242.85                                        121.76
  Modelo mínimo (Regresión Lineal)                           176.60                                        143.43
Modelo alternativo (Random Forest)                           159.11                                        131.23


## 13. Volcado de evidencia

Se guarda un CSV con las métricas y la huella SHA-256 del test compartido por los tres procedimientos, para que el Integrante 4 lo consuma al armar la tabla comparativa y para dejar trazabilidad de que los tres corrieron sobre el mismo conjunto.

In [14]:
evidencia = pd.DataFrame({
    "procedimiento": resultados["procedimiento"],
    "n_test": len(test),
    "huella_test_sha256_16": huella_test_final,
    "mae": resultados["mae"],
    "rmse": resultados["rmse"],
    "pct_subestimaciones": resultados["pct_subestimaciones"],
    "mae_subestimacion": resultados["mae_subestimacion"],
    "mae_sobrestimacion": resultados["mae_sobrestimacion"],
    "mejora_pct_sobre_ingeniero": [0.0, round(mejora_lin, 2), round(mejora_rf, 2)],
})
evidencia.to_csv("evidencia_tres_procedimientos.csv", index=False)
print("Guardado: evidencia_tres_procedimientos.csv")
evidencia

Guardado: evidencia_tres_procedimientos.csv


,procedimiento,n_test,huella_test_sha256_16,mae,rmse,pct_subestimaciones,mae_subestimacion,mae_sobrestimacion,mejora_pct_sobre_ingeniero
0,Baseline operativo (ingeniero),1273,40e463d204777e03,121.76,238.54,0.436,27.21,195.89,0.00
1,Modelo mínimo (Regresión Lineal),1273,40e463d204777e03,143.43,159.24,0.610,150.93,131.71,-17.80
2,Modelo alternativo (Random Forest),1273,40e463d204777e03,131.23,169.25,0.764,162.35,30.75,-7.78
